[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tymworld/DROPPS/blob/v1.0.0/examples/colab/00_DROPPS_1_0_Quickstart.ipynb)

# DROPPS 1.0 Colab Quickstart

This notebook performs a complete, deliberately tiny workflow: sequence →
coarse-grained model → multichain box → portable DROPPS TPR v2 → short
OpenMM run → validation → density analysis. It is an interface and file-flow
check, not a phase-separation production simulation.

Use the focused notebooks for multicomponent systems, structural restraints,
GPU/restart controls, trajectory transformations, and manuscript-level analyses.


## Runtime

In Colab choose **Runtime → Change runtime type → GPU** when testing CUDA.
This quickstart defaults to OpenMM's deterministic `Reference` platform so it
also works without a GPU.


In [ ]:
#@title Install DROPPS 1.0
#@markdown The default route installs the immutable wheel attached to the
#@markdown public GitHub `v1.0.0` release. Upload remains available for offline use.
installation_source = "Install from the DROPPS v1.0.0 GitHub release" #@param ["Install from the DROPPS v1.0.0 GitHub release", "Upload the DROPPS 1.0 wheel", "Install from another wheel URL"]
wheel_url = "https://github.com/tymworld/DROPPS/releases/download/v1.0.0/dropps-1.0-py3-none-any.whl" #@param {type:"string"}
accelerator_dependencies = "auto" #@param ["auto", "cuda12", "cuda13", "latest"]

import importlib.metadata
import re
import shutil
import subprocess
import sys
from pathlib import Path


def detect_install_extra():
    if accelerator_dependencies != "auto":
        return accelerator_dependencies
    if shutil.which("nvidia-smi"):
        probe = subprocess.run(
            ["nvidia-smi"], text=True, capture_output=True, check=False
        ).stdout
        match = re.search(r"CUDA Version:\s*(\d+)", probe)
        if match and int(match.group(1)) >= 13:
            return "cuda13"
        return "cuda12"
    return "latest"


if installation_source.startswith("Upload"):
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "This upload form is intended for Google Colab. Set installation_source "
            "to the URL option when running elsewhere."
        ) from exc
    uploaded = files.upload()
    wheel_candidates = [Path(name) for name in uploaded if name.endswith(".whl")]
    if len(wheel_candidates) != 1:
        raise ValueError("Upload exactly one DROPPS 1.0 .whl file.")
    wheel_target = str(wheel_candidates[0].resolve())
else:
    if not wheel_url.strip():
        raise ValueError("Provide the published DROPPS 1.0 wheel URL.")
    wheel_target = wheel_url.strip()

extra = detect_install_extra()
if wheel_target.startswith(("https://", "http://")):
    package_spec = f"dropps[{extra}] @ {wheel_target}"
else:
    package_spec = f"{wheel_target}[{extra}]"
print(f"Installing {package_spec}")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        package_spec,
    ],
    check=True,
)

version = importlib.metadata.version("dropps")
if version != "1.0":
    raise RuntimeError(f"Expected DROPPS 1.0, but installed {version}.")
subprocess.run(["dps", "--version"], check=True)


In [ ]:
#@title Shared notebook helpers
import hashlib
import json
import os
import shlex
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

COMMAND_LOG = []


def run_command(arguments, *, cwd=None, input_text=None, check=True):
    command = [str(value) for value in arguments]
    print("$", shlex.join(command))
    result = subprocess.run(
        command,
        cwd=None if cwd is None else str(cwd),
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    COMMAND_LOG.append(
        {
            "time_utc": datetime.now(timezone.utc).isoformat(),
            "cwd": str(Path(cwd or Path.cwd()).resolve()),
            "command": command,
            "returncode": result.returncode,
        }
    )
    if check and result.returncode:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {shlex.join(command)}"
        )
    return result


def dps(*arguments, cwd=None, input_text=None, check=True):
    return run_command(
        ["dps", *arguments], cwd=cwd, input_text=input_text, check=check
    )


def reset_task_directory(path):
    path = Path(path).resolve()
    if not path.name.startswith("dropps_"):
        raise ValueError(f"Refusing to reset unexpected directory: {path}")
    if path.exists():
        import shutil

        shutil.rmtree(path)
    path.mkdir(parents=True)
    return path


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract_zip(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        bundle.extractall(destination)


def find_unique(root, basename):
    matches = [path for path in Path(root).rglob(basename) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one file named {basename!r} below {root}, found {len(matches)}."
        )
    return matches[0]


def make_zip(paths, output_path, *, base=None):
    output_path = Path(output_path)
    base = Path(base or output_path.parent).resolve()
    with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for source in sorted({Path(path).resolve() for path in paths}):
            if source == output_path.resolve() or not source.is_file():
                continue
            try:
                arcname = source.relative_to(base)
            except ValueError:
                arcname = Path(source.name)
            bundle.write(source, arcname)
    print(f"Wrote {output_path} ({output_path.stat().st_size / 1024:.1f} KiB)")
    return output_path


def download(path):
    try:
        from google.colab import files
    except ImportError:
        print(f"Result available at {Path(path).resolve()}")
    else:
        files.download(str(path))


def optional_time_arguments(start, end, interval):
    arguments = []
    for option, value in (("-b", start), ("-e", end), ("-dt", interval)):
        if str(value).strip():
            arguments.extend([option, str(value).strip()])
    return arguments


def read_xvg(path):
    legends = {}
    metadata = {}
    rows = []
    with Path(path).open(encoding="utf-8") as stream:
        for raw in stream:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("@"):
                re_module = __import__("re")
                match = re_module.search(r's(\d+)\s+legend\s+"(.*)"', line)
                if match:
                    legends[int(match.group(1)) + 1] = match.group(2)
                for key, pattern in {
                    "title": r'@\s+title\s+"(.*)"',
                    "xlabel": r'@\s+xaxis\s+label\s+"(.*)"',
                    "ylabel": r'@\s+yaxis\s+label\s+"(.*)"',
                }.items():
                    label_match = re_module.search(pattern, line)
                    if label_match:
                        metadata[key] = label_match.group(1)
                continue
            rows.append([float(value) for value in line.split()])
    data = np.asarray(rows, dtype=float)
    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(f"No plottable data in {path}")
    return data, legends, metadata


def plot_xvg(path, *, title=None, output=None):
    data, legends, metadata = read_xvg(path)
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for column in range(1, data.shape[1]):
        ax.plot(data[:, 0], data[:, column], label=legends.get(column, f"series {column}"))
    ax.set_title(title or metadata.get("title", Path(path).stem))
    ax.set_xlabel(metadata.get("xlabel", "x / time"))
    ax.set_ylabel(metadata.get("ylabel", "value"))
    if data.shape[1] > 2:
        ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    output_path = Path(output) if output is not None else Path(path).with_suffix(".png")
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    print(f"Saved figure: {output_path}")
    plt.show()
    return fig, output_path


In [ ]:
#@title Configure the tiny demonstration
sequence = "FWFWFWFWFWFWFWFW" #@param {type:"string"}
molecule_name = "PEP" #@param {type:"string"}
forcefield = "HPS" #@param ["HPS", "HPST", "CALVADOS2", "HPSRNA", "MPiPi", "MPiPi_PTM"]
molecule_count = 4 #@param {type:"integer"}
random_seed = 1215 #@param {type:"integer"}
validation_steps = 20 #@param {type:"integer"}
validation_platform = "Reference" #@param ["Reference", "CPU", "CUDA", "OpenCL", "auto"]

BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
WORKDIR = reset_task_directory(BASE / "dropps_quickstart")
print("Working directory:", WORKDIR)


## 1 Build a molecule and a small system


In [ ]:
dps(
    "pdb2dps", "-s", sequence, "-ff", forcefield,
    "-oc", f"{molecule_name}.pdb", "-op", f"{molecule_name}.itp",
    "-on", molecule_name, "--seed", random_seed,
    cwd=WORKDIR,
)
dps(
    "genmesh", "-f", f"{molecule_name}.pdb", "-p", f"{molecule_name}.itp",
    "-n", molecule_count, "-mesh", 2, 2, 1, "-g", 1,
    "-mx", 4, "-my", 4, "-mz", 4,
    "-oc", "system.pdb", "-op", "system.top", "--seed", random_seed,
    cwd=WORKDIR,
)


## 2 Compile a portable run input


In [ ]:
mdp = f"""# Tiny DROPPS 1.0 validation run
integrator = Langevin
dt = 0.01
nsteps = {validation_steps}
friction = 1.0
seed = {random_seed}
minimize = True
forcetol = 100
max-step = 100
production-temperature = 300
gen-vel = True
initial-temperature = 300
warming-speed = 1.0
keep-warming-trajectory = False
bondtype = bond
comm-mode = Linear
nstcomm = 2
vdwtype = pLJ
cutoff-scheme-lj = static
cutoff-lj = 1.0
cutoff-lj-multi = 3.0
shift-lj = True
coulombtype = yukawa
cutoff-coul = 1.0
shift-coul = True
salt-conc = 0.1
pcoulp = False
ref-P = 1
tau-P = 5
nst-xout = 5
nst-screenlog = 0
nst-filelog = 5
nst-energy = 5
energy-grps = potentialEnergy,kineticEnergy,totalEnergy,temperature,boxX,boxY,boxZ,volume,density
nst-stress = 0
"""
(WORKDIR / "validation.mdp").write_text(mdp, encoding="utf-8")
dps(
    "grompp", "-f", "system.pdb", "-p", "system.top",
    "-m", "validation.mdp", "-o", "validation.tpr", cwd=WORKDIR,
)


## 3 Run, check, and analyze


In [ ]:
dps(
    "mdrun", "-s", "validation.tpr", "-o", "validation",
    "--platform", validation_platform, "-cpt", 0, cwd=WORKDIR,
)
dps("check", "-s", "validation.tpr", "-f", "validation.xtc", cwd=WORKDIR)
dps(
    "density", "-s", "validation.tpr", "-f", "validation.xtc",
    "-o", "density.xvg", "-selfit", 0, "-sel", 0,
    "--center-mode", "none", cwd=WORKDIR,
)
plot_xvg(WORKDIR / "density.xvg", title="Tiny-run mass-density profile")


## 4 Record provenance and download


In [ ]:
manifest = {
    "notebook": "00_DROPPS_1_0_Quickstart.ipynb",
    "dropps_version": "1.0",
    "purpose": "interface and file-flow validation only",
    "parameters": {
        "sequence": sequence,
        "molecule_name": molecule_name,
        "forcefield": forcefield,
        "molecule_count": molecule_count,
        "random_seed": random_seed,
        "validation_steps": validation_steps,
        "validation_platform": validation_platform,
    },
    "commands": COMMAND_LOG,
}
(WORKDIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
outputs = [path for path in WORKDIR.iterdir() if path.is_file()]
archive = make_zip(outputs, WORKDIR / "DROPPS_1_0_quickstart_results.zip", base=WORKDIR)
download(archive)
